**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**06 | Validation Selection, Robustness and Conditional Held-Out Evaluation**

![Validation selection and development-mode evaluation workflow](../../images/06_robustness_and_final_evaluation.svg)

**Research objective:** Reconcile the seven classical candidates from Notebook 04 and three compact GRU candidates from Notebook 05 on the same source-aligned Validation windows. Freeze the primary candidate using **fixed-label Validation Macro F1**, then Balanced Accuracy and a prespecified model priority.

**Recorded inputs:** ZuCo 2.0 (NEMAR `nm000279`, release `v1.0.0`); **206 paired 12-second windows** represented by `02_window_registry.csv`, `03_features.csv` and `03_temporal_sequences.npz`. Train consists of 95 windows from three people, Validation of 31 windows from one person and the reserved Test of 80 windows from two different people. Saved `04_validation_*.csv` and `05_validation_*.csv` cover all ten candidates and the identical 31 Validation IDs.

**Observed before this notebook runs:** Gaze logistic regression has the highest recorded Validation Macro F1 (**0.8862**) across ten models; the strongest temporal candidate, EEG GRU, has **0.6890**. Notebook 06 will verify these upstream predictions and apply the prespecified ranking before recording its own selected model. These are exploratory results from a single Validation participant.

**Development execution:** Keep `RUN_FINAL_TEST=False`. This produces `06_model_selection.csv`, `06_validation_sensitivity.csv` and `06_validation_evidence.png`, checking the fixed early-fusion model under EEG/gaze feature masking on **Validation only**. **No Test predictions, scores, confusion matrix or participant-level Test estimates are authorised in this run.**

**Conditional final evaluation:** The Test-specific branch exists for a separately reviewed and frozen protocol. Do not enable it merely because this notebook runs. The dataset's NR/TSR task conditions do not directly measure comprehension, intelligence or clinical outcomes. A single Validation person and two reserved Test people cannot support population-level reliability claims.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Isolated Repository Checkout ------------------
from pathlib import Path
import subprocess
import sys

repo_url = (
    "https://github.com/RezaSaadatyar/"
    "Eye-Tracking-Neurocognitive-Intelligence.git"
)
# Use a separate checkout so stale or dirty Stage-01–05 working files are
# preserved without blocking access to the latest committed Stage-06 inputs.
repo_path = Path("/content/Eye-Tracking-Neurocognitive-Intelligence-stage06")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", "--branch", "main", "--single-branch",
         repo_url, str(repo_path)], check=True
    )
elif not (repo_path / ".git").is_dir():
    raise RuntimeError("The Stage-06 checkout path exists but is not a Git repository")

# Avoid mixing Python modules previously loaded from another checkout.
for module_name in (
    "features", "models", "core.plot_style", "core.output_download",
    "python_modules.core.colab_setup",
    "python_modules.core.repository_sync",
):
    loaded = sys.modules.get(module_name)
    module_file = getattr(loaded, "__file__", None)
    if module_file and not Path(module_file).resolve().is_relative_to(
        repo_path.resolve()
    ):
        raise RuntimeError(
            "A research module from a different checkout is still loaded. "
            "Start Notebook 06 in a fresh Colab Runtime to avoid stale imports."
        )

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import subprocess
import sys
from pathlib import Path

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 5. Machine Learning and Deep Learning -------------------
import joblib
from sklearn.metrics import ConfusionMatrixDisplay
import torch

# ------------------- 6. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 7. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from features import verify_temporal_feature_alignment
from models import (
    MODEL_ORDER, CLASSICAL_ORDER, NEURAL_ORDER, CLASS_TO_INDEX,
    metrics, matrix, probabilities, record_predictions,
    torch_network, predict_neural, feature_columns,
)

# ---------------------------- 8. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 9. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


**02. Reconcile Stage-02–05 Validation Evidence**

Load the Stage-03 predictor table, cross-check its original `window_id`, participant, task and partition against the eligible Stage-02 registry, and confirm that each person belongs to only one split. Verify **Train/Validation temporal arrays** against the saved feature means, recompute **Train-only scaler statistics** for comparison with `05_scaling.npz`, and inspect the three GRU checkpoint metadata and tensor shapes **without training or scoring**. Reconcile the saved `04_*` and `05_*` Validation predictions, class probabilities and metrics for **all ten models**.

Every candidate must use exactly the same **31 Validation IDs and NR/TSR labels**. Reject missing models, duplicates, unexpected partitions, invalid probability pairs and disagreement between saved and recomputed metrics. This stage must not calculate any Test performance.


In [ ]:
# ========= 02. Load Upstream Models and Reconcile Validation Records =========

# ----------------------- 1. Source Role and Identity Audit -------------------
table = pd.read_csv(
    output_dir / '03_features.csv', dtype={'participant_id': str})
registry = pd.read_csv(
    output_dir / '02_window_registry.csv',
    dtype={'participant_id': str})
if table.empty or table.window_id.duplicated().any():
    raise ValueError('Missing or duplicated Stage-03 feature identities')
if not table.split.isin(('train', 'validation', 'test')).all():
    raise ValueError('Unexpected Stage-03 split role')
if not table.task.isin(CLASS_TO_INDEX).all():
    raise ValueError('Unexpected Stage-03 task label')
eligible = registry.loc[
    registry.eligible.eq(True),
    ['window_id', 'participant_id', 'task', 'split']]
if eligible.window_id.duplicated().any():
    raise ValueError('Duplicated eligible Stage-02 window identities')
if set(table.window_id) != set(eligible.window_id):
    raise ValueError('Stage-02 and Stage-03 window identities differ')
matched = table[
    ['window_id', 'participant_id', 'task', 'split']
].merge(
    eligible, on='window_id', validate='one_to_one',
    suffixes=('_03', '_02'))
for field in ('participant_id', 'task', 'split'):
    if not matched[f'{field}_03'].eq(matched[f'{field}_02']).all():
        raise ValueError(f'Stage-02/03 {field} mismatch')
if not table.groupby('participant_id').split.nunique().eq(1).all():
    raise ValueError('Participant overlap between Train, Validation and Test')

valid = table.loc[table.split.eq('validation')].copy()
y_valid = valid.task.map(CLASS_TO_INDEX).to_numpy(dtype=int)
test_n_people = table.loc[
    table.split.eq('test'), 'participant_id'].nunique()
if (valid.empty or valid.task.nunique() != 2 or
        valid.participant_id.nunique() < 1 or test_n_people < 1):
    raise ValueError('Missing Validation classes or reserved Test roles')

# ------------- 2. Temporal Inputs and Train-Fitted Scaler QA ---------------
# Check only Train/Validation sequence rows here; do not score Test.
dev_idx = np.where(table.split.ne('test').to_numpy())[0]
train_idx = np.where(table.split.eq('train').to_numpy())[0]
if len(train_idx) < 1:
    raise ValueError('Training partition is empty')
with np.load(output_dir / '03_temporal_sequences.npz',
             allow_pickle=False) as temporal:
    development = {
        key: temporal[key][dev_idx] for key in ('eeg', 'gaze')
    }
    training = {
        key: temporal[key][train_idx] for key in ('eeg', 'gaze')
    }
verify_temporal_feature_alignment(
    development, table.iloc[dev_idx].reset_index(drop=True))
with np.load(output_dir / '05_scaling.npz',
             allow_pickle=False) as recorded_scale:
    for key, sequence in training.items():
        flat = sequence.reshape(-1, sequence.shape[-1])
        median = np.nanmedian(flat, axis=0)
        median = np.where(np.isfinite(median), median, 0.)
        filled = np.where(np.isfinite(flat), flat, median)
        expected_scale = {
            'median': median.astype('float32'),
            'mean': filled.mean(axis=0).astype('float32'),
            'std': np.maximum(
                filled.std(axis=0), 1e-6).astype('float32'),
        }
        for name, expected in expected_scale.items():
            recorded = recorded_scale[f'{key}_{name}']
            if (recorded.shape != expected.shape or
                    not np.isfinite(recorded).all() or
                    not np.allclose(
                        recorded, expected, rtol=1e-5, atol=1e-5)):
                raise ValueError(
                    f'Stage-05 {key}_{name} disagrees with '
                    'Train-derived Stage-03 sequences')
print('Train/Validation temporal rows and Stage-05 scaling verified')

# --------------------- 3. Candidate Table Consistency ------------------------
source = pd.concat([
    pd.read_csv(output_dir / '04_validation_predictions.csv',
                dtype={'participant_id': str}),
    pd.read_csv(output_dir / '05_validation_predictions.csv',
                dtype={'participant_id': str})],
    ignore_index=True)
summaries = pd.concat([
    pd.read_csv(output_dir / '04_validation_metrics.csv'),
    pd.read_csv(output_dir / '05_validation_metrics.csv')],
    ignore_index=True)
if (len(summaries) != len(MODEL_ORDER) or
        summaries.model.duplicated().any() or
        set(summaries.model) != set(MODEL_ORDER) or
        set(source.model) != set(MODEL_ORDER)):
    raise ValueError('Missing, duplicated or unexpected model candidates')
if not source.split.eq('validation').all():
    raise ValueError('Non-Validation rows in the candidate predictions')
if not summaries.partition.eq('validation').all():
    raise ValueError('Non-Validation rows in candidate metrics')
if source.duplicated(['model', 'window_id']).any():
    raise ValueError('Duplicate model/window prediction records')

# -------------------- 4. Saved Temporal Checkpoints QA -----------------------
# Inspect checkpoint metadata and state shapes, without scoring any data.
for name in NEURAL_ORDER:
    checkpoint_path = output_dir / f'05_{name}.pt'
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=True)
    stored = summaries.loc[summaries.model.eq(name)].iloc[0]
    if (checkpoint.get('model') != name or
            int(checkpoint['best_epoch']) != int(stored.best_epoch)):
        raise ValueError(f'Stage-05 checkpoint metadata mismatch: {name}')
    network = torch_network(
        name, development['eeg'].shape[-1],
        development['gaze'].shape[-1],
        hidden=int(checkpoint['hidden']))
    network.load_state_dict(checkpoint['state_dict'], strict=True)
    count = sum(param.numel() for param in network.parameters())
    if count != int(stored.parameters):
        raise ValueError(f'Stage-05 parameter count mismatch: {name}')
print('Three saved temporal checkpoints match current model definitions')

# ------------------------ 5. Aligned Prediction Audit ------------------------
for name in MODEL_ORDER:
    part = source.loc[source.model.eq(name)]
    if len(part) != len(valid) or set(part.window_id) != set(valid.window_id):
        raise ValueError(f'Validation window IDs mismatch: {name}')
    aligned = part.set_index('window_id').loc[valid.window_id]
    for field in ('participant_id', 'task', 'split'):
        if not np.array_equal(
                aligned[field].to_numpy(),
                valid[field].to_numpy()):
            raise ValueError(f'Validation {field} mismatch: {name}')
    if not np.array_equal(
            aligned.target_index.to_numpy(dtype=int), y_valid):
        raise ValueError(f'Validation true labels differ: {name}')
    proba = aligned[['p_nr', 'p_tsr']].to_numpy(dtype=float)
    if (not np.isfinite(proba).all() or
            (proba < 0).any() or (proba > 1).any() or
            not np.allclose(proba.sum(axis=1), 1, atol=1e-5) or
            not np.array_equal(aligned.pred_index.to_numpy(dtype=int),
                               proba.argmax(axis=1))):
        raise ValueError(f'Invalid Validation probabilities: {name}')
    recorded = summaries.loc[summaries.model.eq(name)].iloc[0]
    if (int(recorded.n_windows) != len(valid) or
            int(recorded.n_people) != valid.participant_id.nunique()):
        raise ValueError(f'Validation support mismatch: {name}')
    recomputed = metrics(
        y_valid, aligned.pred_index.to_numpy(dtype=int))
    for metric, score in recomputed.items():
        if not np.isclose(score, recorded[metric], atol=1e-9):
            raise ValueError(f'Saved {metric} differs: {name}')
print('Reconciled', len(MODEL_ORDER), 'models on', len(valid),
      'Validation windows from', valid.participant_id.nunique(),
      'independent person(s). Test remains unscored.')


**03. Freeze the Prespecified Validation-Selected Candidate**

Rank all ten candidates by **fixed-label Validation Macro F1**, then Balanced Accuracy, and finally the fixed model priority defined in `models.py`. Record the selected model and its Validation support before any optional Test branch.

Based on the saved Stage-04/05 evidence, gaze logistic regression currently leads at Macro F1 **0.8862**. The notebook recalculates and verifies this ranking rather than treating a reported value as an automatic selection. The ranking must never use Test outcomes.


In [ ]:
# =========== 03. Freeze the Validation-Selected Primary Candidate ============

# ------------------------- 1. Primary Ranking Rule ---------------------------
ranking = summaries[['model', 'macro_f1', 'balanced_accuracy']].copy()
ranking['priority'] = ranking.model.map(MODEL_ORDER.index)
ranking = ranking.sort_values(
    ['macro_f1', 'balanced_accuracy', 'priority'],
    ascending=[False, False, True]).reset_index(drop=True)
selected_name = str(ranking.iloc[0].model)
protocol = json.loads((output_dir / '01_protocol.json').read_text())
if protocol.get('target') != CLASS_TO_INDEX:
    raise ValueError('Source protocol labels differ from fixed NR/TSR order')

# --------------------------- 2. Test Policy Guard ----------------------------
RUN_FINAL_TEST = False  # Preserve the held-out Test in development mode.
evaluation_mode = 'final' if RUN_FINAL_TEST else 'development'
if not RUN_FINAL_TEST:
    prior_test_files = [
        name for name in (
            '06_test_metrics.csv', '06_test_predictions.csv',
            '06_person_metrics.csv', '06_cluster_bootstrap.csv',
            '06_primary_test_confusion.png'
        ) if (output_dir / name).exists()
    ]
    if prior_test_files:
        raise RuntimeError(
            'Previous Test artefacts exist in this checkout. '
            'Use a clean development checkout; existing evidence was '
            'preserved and no Test results were opened: '
            + ', '.join(prior_test_files)
        )

# -------------------- 3. Save Validation-Only Decision -----------------------
decision = pd.DataFrame([{
    'selected_model': selected_name,
    'validation_macro_f1': float(ranking.iloc[0].macro_f1),
    'validation_n_people': valid.participant_id.nunique(),
    'test_used_in_selection': False,
    'run_mode': evaluation_mode,
    'source_people': table.participant_id.nunique(),
    'dataset_version': protocol['dataset_version'],
}])
decision.to_csv(output_dir / '06_model_selection.csv', index=False)
display(ranking)
print('Frozen Validation primary:', selected_name)
print('Test scoring enabled:', RUN_FINAL_TEST)


**04. Conditional Test Scoring (Not Run in Development Mode)**

This branch exists for a separately approved, frozen final evaluation. **Do not change `RUN_FINAL_TEST=False` for this development run.** When disabled, no Test features are passed to a model, no Test labels are scored and no Test output files are created.

The two reserved Test participants are not external clinical validation; this remains a limited same-source participant-disjoint cohort.


In [ ]:
# ========= 04. Score Frozen Models on the Test Only When Authorised ==========

test_predictions, test_scores = [], []
if RUN_FINAL_TEST:
    # Test rows and labels are only accessed inside the authorised branch.
    test = table.loc[table.split.eq('test')].copy()
    y_test = test.task.map(CLASS_TO_INDEX).to_numpy()
    scaling = np.load(output_dir / '05_scaling.npz',
                      allow_pickle=False)
    sequences = np.load(output_dir / '03_temporal_sequences.npz',
                        allow_pickle=False)
    verify_temporal_feature_alignment(sequences, table)
    test_idx = np.where(table.split.eq('test').to_numpy())[0]

    def standardised(key):
        original = sequences[key]
        median = scaling[f'{key}_median']
        mean = scaling[f'{key}_mean']
        std = scaling[f'{key}_std']
        return ((np.where(np.isfinite(original), original, median)
                 - mean) / std).astype('float32')

    eeg_test = standardised('eeg')[test_idx]
    gaze_test = standardised('gaze')[test_idx]
    probas = {}
    for name in CLASSICAL_ORDER:
        if name == 'late_equal_fusion':
            continue
        group = ('eeg' if name.startswith('eeg_') else 'gaze'
                 if name.startswith('gaze_') or name == 'class_prior'
                 else 'fused')
        fitted = joblib.load(output_dir / f'04_{name}.joblib')
        probas[name] = probabilities(fitted, matrix(test, group))
    probas['late_equal_fusion'] = 0.5 * (
        probas['eeg_logistic'] + probas['gaze_logistic'])
    for name in NEURAL_ORDER:
        checkpoint = torch.load(output_dir / f'05_{name}.pt',
                                map_location='cpu',
                                weights_only=True)
        net = torch_network(name, eeg_test.shape[-1],
                            gaze_test.shape[-1],
                            hidden=int(checkpoint['hidden']))
        net.load_state_dict(checkpoint['state_dict'])
        probas[name] = predict_neural(net, eeg_test, gaze_test)
    for name in MODEL_ORDER:
        score = metrics(y_test, probas[name].argmax(axis=1))
        test_scores.append({
            'model': name, 'n_windows': len(test),
            'n_people': test.participant_id.nunique(),
            'selected_primary': name == selected_name, **score})
        test_predictions.append(record_predictions(test, name,
                                                   probas[name]))
    test_metrics = pd.DataFrame(test_scores)
    prediction_table = pd.concat(test_predictions, ignore_index=True)
    test_metrics.to_csv(output_dir / '06_test_metrics.csv',
                        index=False)
    prediction_table.to_csv(output_dir / '06_test_predictions.csv',
                            index=False)
    display(test_metrics.sort_values('macro_f1', ascending=False))
else:
    print('Development mode: no Test scoring performed.')


**05. Person-Level Test Support and Uncertainty (Conditional)**

Only a separately authorised final Test evaluation may generate per-person metrics or participant-cluster bootstrap summaries. With **two reserved Test participants** in the current development cohort, no credible participant-cluster bootstrap confidence interval is produced. This notebook skips the uncertainty branch entirely while `RUN_FINAL_TEST=False`.


In [ ]:
# ======= 05. Audit Person-Level Test Support and Clustered Uncertainty =======

if RUN_FINAL_TEST:
    groupwise = []
    for name, part in prediction_table.groupby('model'):
        for person, records in part.groupby('participant_id'):
            groupwise.append({'model': name,
                              'participant_id': person,
                              'n_windows': len(records),
                              **metrics(records.target_index,
                                        records.pred_index)})
    grouped = pd.DataFrame(groupwise)
    grouped.to_csv(output_dir / '06_person_metrics.csv', index=False)
    primary = prediction_table.loc[
        prediction_table.model.eq(selected_name)].copy()
    people = primary.participant_id.unique()
    ci = []
    if len(people) >= 3:
        rng = np.random.default_rng(SEED)
        for repeat in range(500):
            sampled = rng.choice(people, len(people), replace=True)
            samples = pd.concat([
                primary.loc[primary.participant_id.eq(p)]
                for p in sampled], ignore_index=True)
            ci.append({'bootstrap_id': repeat,
                       **metrics(samples.target_index,
                                 samples.pred_index)})
    uncertainty = pd.DataFrame(ci, columns=[
        'bootstrap_id', 'accuracy', 'macro_f1',
        'balanced_accuracy'])
    uncertainty.to_csv(output_dir / '06_cluster_bootstrap.csv',
                       index=False)
    if not uncertainty.empty:
        print('Cluster bootstrap Macro F1 95% interval:',
              np.quantile(uncertainty.macro_f1, [0.025, 0.975]))
    else:
        print('Too few independent Test people for bootstrap CI.')
else:
    print('No groupwise Test analysis in development mode.')


**06. Validation Sensitivity of the Fixed Early-Fusion Model**

As a **Validation-only descriptive perturbation**, set all EEG or all gaze input features to missing values and let the fitted Stage-04 imputer replace them with its Train-derived medians. Recompute early-fusion logistic predictions without refitting or retuning.

This is not an attribution method, a controlled physiological ablation or a measure of model reliability across people. The original model remains unchanged and these sensitivity scores do not alter the prespecified primary selection.


In [ ]:
# ======== 06. Inspect Validation Sensitivity Without Retuning Models =========

model = joblib.load(
    output_dir / '04_early_fusion_logistic.joblib')
columns = feature_columns(valid, 'fused')
original = valid[columns].copy()
observations = []
for change in ('unchanged', 'mask_eeg', 'mask_gaze'):
    x = original.copy()
    if change != 'unchanged':
        prefix = 'eeg_' if change == 'mask_eeg' else 'gaze_'
        x.loc[:, [col for col in columns if col.startswith(prefix)]] = np.nan
    prediction = probabilities(model, x.to_numpy(dtype=float))
    observations.append({
        'condition': change, 'model': 'early_fusion_logistic',
        'partition': 'validation', 'n_windows': len(valid),
        'n_people': valid.participant_id.nunique(),
        **metrics(y_valid, prediction.argmax(axis=1))})
robustness = pd.DataFrame(observations)
robustness.to_csv(output_dir / '06_validation_sensitivity.csv',
                  index=False)
display(robustness)


**07. Save Validation Evidence and Conditional Figures**

In the initial development run, generate a figure showing all ten verified Validation Macro F1 values and the three `early_fusion_logistic` Validation sensitivity conditions. The figure and tables document observed evidence from **one Validation person**.

A Test confusion matrix, Test metric tables and participant-level Test summaries are generated **only** under a separately authorised final-evaluation run. Do not interpret the development figure as a final Test result.


In [ ]:
# ======= 07. Create Evaluation Figures and Verify Stage-Owned Outputs ========

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ranking.sort_values('macro_f1').plot.barh(
    x='model', y='macro_f1', ax=axes[0], legend=False)
axes[0].set(xlabel='Macro F1', title='Validation model selection',
            xlim=(0, 1.03), ylabel='Candidate')
axes[1].bar(robustness.condition, robustness.macro_f1)
axes[1].set(xlabel='Fixed-feature perturbation',
            ylabel='Validation Macro F1', ylim=(0, 1.05),
            title='Early-fusion sensitivity (not attribution)')
axes[1].tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(output_dir / '06_validation_evidence.png', dpi=170)
plt.show()
if RUN_FINAL_TEST:
    primary = prediction_table.loc[
        prediction_table.model.eq(selected_name)]
    fig, ax = plt.subplots(figsize=(4.6, 4.3))
    ConfusionMatrixDisplay.from_predictions(
        primary.target_index, primary.pred_index,
        labels=[0, 1], display_labels=['NR', 'TSR'],
        ax=ax, colorbar=False)
    ax.set_title('Primary held-out Test confusion')
    fig.tight_layout()
    fig.savefig(output_dir / '06_primary_test_confusion.png',
                dpi=170)
    plt.show()
    assert prediction_table.groupby('model').window_id.nunique().eq(
        len(test)).all()
    assert set(test_metrics.model) == set(MODEL_ORDER)
    print('Saved main Test evidence for',
          test.participant_id.nunique(), 'independent people.')
else:
    print('Final Test outputs are deferred until RUN_FINAL_TEST=True.')


**08. Interpretation and Next Steps**

After execution, verify that `06_model_selection.csv`, `06_validation_sensitivity.csv` and `06_validation_evidence.png` were produced, and that no `06_test_*` result was generated. Record the selected candidate based on the verified Stage-04/05 Validation evidence and interpret the feature-masking analysis conservatively.

Important limits include **one independent Validation participant**, limited source-person coverage, unequal reading-task stimuli, possible eye artefacts in EEG, and **metadata-based rather than independently drift-verified** EEG/gaze alignment. Neither reading-task classification nor a learned fusion gate establishes comprehension, cognition, clinical utility or causal physiology. **Final Test scoring remains a separate decision, not an automatic next step.**


In [ ]:
# =============== 08. Final Interpretation and Reproducibility ================

assert (output_dir / '06_model_selection.csv').exists()
assert (output_dir / '06_validation_sensitivity.csv').exists()
if RUN_FINAL_TEST:
    assert (output_dir / '06_test_metrics.csv').exists()
    print('Stage 06 final evaluation files are ready for review.')
else:
    print('Development evidence only. Keep final Test untouched.')
print('Scientific report:',
      work / 'RESEARCH_METHODS_AND_EVIDENCE.md')

# ---------------------- 2. Package Stage-Owned Outputs -----------------------
download_outputs_from_colab(
    "eeg_eye_tracking_fusion/output", "06"
)
